In [9]:
###############################################################
# VectorStore, Retriever 준비
###############################################################
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

COLLECTION_NAME = "olympic_info"
PERSIST_DIRECTORY = "vector_store/chroma/olympic_info"

def get_vectorstore():
    embedding_model = OpenAIEmbeddings(
        model="text-embedding-3-large",
    )

    vector_store = Chroma(
        embedding_function=embedding_model,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIRECTORY
    )
    return vector_store


def get_retriever(k=10):
    vector_store = get_vectorstore()
    retriever = vector_store.as_retriever(search_kwargs={"k":k})
    return retriever

In [10]:
###########################################################################
# Document 추가
###########################################################################

from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

vector_store = get_vectorstore()

path = 'data/olympic.txt'
loader = TextLoader(path, encoding="utf-8")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50
)

docs = loader.load_and_split(splitter)

ids = vector_store.add_documents(documents=docs)
len(ids)

61

In [5]:
retriever = get_retriever(k=3)
retriever.invoke("IOC에 대해서 설명해줘")

[Document(id='6334f886-81f0-4442-bd85-2f815adb0bf7', metadata={'source': 'data/olympic.txt'}, page_content='국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다. 국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동은 크게 세 가지로 구성된다.\n- 국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구이다. 예를 들어서 국제 축구 연맹(FIFA)는 축구를 주관하며, 국제 배구 연맹(FIVB)은 배구를 주관하는 기구이다. 올림픽에는 현재 35개의 국제경기연맹이 있고 각 종목을 대표한다. (이 중에는 올림픽 종목은 아니지만 IOC의 승인을 받은 연맹도 있다.)'),
 Document(id='052719f0-853a-4c2a-a511-c9730f9883f6', metadata={'source': 'data/olympic.txt'}, page_content='국제 올림픽 위원회\n올림픽 활동이란 많은 수의 국가, 국제 경기 연맹과 협회 • 미디어 파트너를 맺기 • 선수, 직원, 심판, 모든 사람과 기관이 올림픽 헌장을 지키는 것을 말한다. 국제올림픽위원회(IOC)는 모든 올림픽 활동을 통솔하는 단체로서, 올림픽 개최 도시 선정, 계획 감독, 종목 변경, 스폰서 및 방송권 계약 체결 등의 권리가 있다. 올림픽 활동은 크게 세 가지로 구성된다.\n- 국제경기연맹(IF)은 국제적인 규모의 경기를 관리, 감독하는 기구이다. 예를 들어서 국제 축구 연맹(FIFA)는 축구를 주관하며, 국제 배구 연맹(FIVB)은 배구를 주관하는 기구이다. 올림픽에는 현재 35개의 국제경기연맹이 있고 각 종목을 대표한다. (이 중에는 올림픽 종목은 아

# Rerank

## 개념

- RAG의 정확도는 관련 정보의 컨텍스트 내 존재 유무가 아니라 순서가 중요하다. 즉, 관련 정보가 컨텍스트 내 상위권에 위치하고 있을 때 좋은 답변을 얻을 수 있다는 뜻이다. 
- **Rerank**는 RAG 시스템에서 **초기 검색 단계에서 추출된 후보 문서들의 순위를 재조정**하는 기법이다. 
- 벡터 유사도 기반의 **빠른 1차 검색** 후, 보다 **정밀한 모델**(예: Cross-encoder, LLM 등)을 활용해 질문과 검색된 문서간의 의미론적 관련성을 평가하여, 실제로 답변 생성에 가장 **적합한 문서들이 상위**에 오도록 순서를 다시 매긴다. 이를 통해 LLM이 더 정확하고 관련성 높은 정보를 바탕으로 답변을 생성할 수 있게 도와준다.

## 방법

- **Cross-encoder 기반 Rerank**  
  - Cross Encoder를 이용해서 순위를 재 지정한다.
  - 1차적으로 얘를 쓰면 연산량이 많아져서 서비스하기 힘듦.
  - Cross-encoder
    - 질문과 문서를 같이 입력으로 받아 둘간의 유사도 점수를 예측하도록 학습한 모델.
    - 학습을 두 문장의 유사도록 예측하도록 학습하였기 때문에 단순 유사도 검사 보다 두 문장간의 의미적 관련성등을 이용해 유사도를 예측하기 때문에 더 정확한 결과를 보인다.
  	- 1차적으로 검색한 문서와 질문간의 유사도를 **cross-encoder**로 다시 계산해서 문서의 순위를 재 조정한다.
  > - **Bi-encoder**
  >     - 질문 (query)와 문서(document)를 각각 독립적으로 인코딩한 후, 벡터 간 유사도 계산
  >     - Encoder 모델은 개별 문장을 입력받아 embedding vector를 출력한다. 질문과 문서를 각각 encoding한 뒤에 둘 간의 유사도를 계산한다.
  
  ![bi_crosss_encoder](figures/bi_cross_encoder.png)

  \[출처:https://aws.amazon.com/ko/blogs/tech/korean-reranker-rag/\]

- **LLM 기반 Rerank**  
  GPT-3, GPT-4 등 LLM을 활용해 각 문서가 질문에 얼마나 부합하는지 평가하여 순위를 매긴다. 성능이 뛰어나지만 비용이 높고 응답 속도가 느릴 수 있다.
## Rerank RAG 프로세스  
1. 1차 검색(예: 임베딩 기반 벡터 검색)으로 상위 k개 문서 추출.  
2. Reranker 모델에 질문-문서 쌍을 입력.  
3. 각 쌍의 관련성 점수 산출 및 재정렬.  
4. 상위 n개 문서를 LLM의 컨텍스트로 전달하여 답변 생성.

## 장단점

- **장점**  
  - Rerank를 적용하면 단순 벡터 유사도 기반 검색보다 훨씬 정교하게 질문과 관련된 정보를 추출할 수 있다. 
  - 실제로 생성되는 답변의 품질이 크게 향상되며, 도메인 특화 정보나 복잡한 질의에도 높은 정확도를 보인다.

- **단점**  
  - Cross-encoder나 LLM 기반 Rerank는 연산량이 많아 실시간 응답이 필요한 대규모 서비스에선 **속도 저하**가 발생할 수 있다.
  - 초기 검색 결과(상위 k개)에만 적용하므로, **1차 검색의 품질이 낮으면** Rerank 효과가 제한적일 수 있다.

## CrossEncoder Reranker 예제

In [4]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [11]:
# cross encoder (embedding) model이 필요행! - cohere : embedding쪽이 성능이 좋음. 근데 오픈소스 없삼
reranker_model_id = "BAAI/bge-reranker-v2-m3"

retriever = get_retriever()		# dafault retriever
reranker = HuggingFaceCrossEncoder(model_name = reranker_model_id)
compressor = CrossEncoderReranker(model = reranker, top_n= 5)		# 압축방식 객체 생성 <- crossencoder 객체
reranker_retriever = ContextualCompressionRetriever(
    base_retriever=retriever,	# 처음 문서를 조회할 retriever
    base_compressor=compressor,	# base retriever가 검색한 문서를 압축하는 알고리즘, (Reranker)
)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\lang_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

In [12]:
query = "올림픽과 관련된 논란들을 알려주세요."
base_result = retriever.invoke(query)
rerank_result = reranker_retriever.invoke(query)

In [13]:
base_result_str = [d.page_content for d in base_result]
rerank_result_str = [d.page_content for d in rerank_result]

In [14]:
print(base_result_str)

['올림픽', '올림픽', '올림픽', '1998년에 몇몇 IOC위원들이 2002년 솔트레이크 시티 동계 올림픽 유치 과정에서 미국에게 미국을 올림픽 개최지로 뽑아달라는 뇌물청탁을 받았다는 것이 폭로되었다. 이에 IOC는 사퇴한 IOC위원 4명과 강제 퇴출된 6명에 대한 조사를 했다. 이 스캔들은 이후에 개최지 선정에서 이와 같은 불미스러운 일이 일어나지 않게 하기 위해서 IOC가 개혁에 착수하도록 하는 긍정적인 역할을 하기도 했다.', '1998년에 몇몇 IOC위원들이 2002년 솔트레이크 시티 동계 올림픽 유치 과정에서 미국에게 미국을 올림픽 개최지로 뽑아달라는 뇌물청탁을 받았다는 것이 폭로되었다. 이에 IOC는 사퇴한 IOC위원 4명과 강제 퇴출된 6명에 대한 조사를 했다. 이 스캔들은 이후에 개최지 선정에서 이와 같은 불미스러운 일이 일어나지 않게 하기 위해서 IOC가 개혁에 착수하도록 하는 긍정적인 역할을 하기도 했다.', '1998년에 몇몇 IOC위원들이 2002년 솔트레이크 시티 동계 올림픽 유치 과정에서 미국에게 미국을 올림픽 개최지로 뽑아달라는 뇌물청탁을 받았다는 것이 폭로되었다. 이에 IOC는 사퇴한 IOC위원 4명과 강제 퇴출된 6명에 대한 조사를 했다. 이 스캔들은 이후에 개최지 선정에서 이와 같은 불미스러운 일이 일어나지 않게 하기 위해서 IOC가 개혁에 착수하도록 하는 긍정적인 역할을 하기도 했다.', "BBC 다큐멘터리인 '파노라마'에서는 '매수된 올림픽'이란 주제로 2004년 8월에 방송을 내보내기도 했다. 이때 이 프로그램에서는 2012년 하계 올림픽의 개최지 선정과 관련된 뇌물에 대해서 조사했다. 이 다큐멘터리에서는 특정 후보 도시가 IOC 위원들에게 뇌물수수하는 것이 가능했다고 주장했으며, 특히 파리 시장이었던 베르트랑 들라노에(Bertrand Delanoë)는 영국의 총리인 토니 블레어와 런던올림픽유치위원회가 입후보 규정을 위반했다고 비난했다. 그는 당시 프랑스 대통령이었던 자크 시라크를 목격자로 내세웠지만 시라크 대통령은 

In [16]:
print(rerank_result_str)

["BBC 다큐멘터리인 '파노라마'에서는 '매수된 올림픽'이란 주제로 2004년 8월에 방송을 내보내기도 했다. 이때 이 프로그램에서는 2012년 하계 올림픽의 개최지 선정과 관련된 뇌물에 대해서 조사했다. 이 다큐멘터리에서는 특정 후보 도시가 IOC 위원들에게 뇌물수수하는 것이 가능했다고 주장했으며, 특히 파리 시장이었던 베르트랑 들라노에(Bertrand Delanoë)는 영국의 총리인 토니 블레어와 런던올림픽유치위원회가 입후보 규정을 위반했다고 비난했다. 그는 당시 프랑스 대통령이었던 자크 시라크를 목격자로 내세웠지만 시라크 대통령은 이 분쟁에 휘말려드는 것을 주의했으며 인터뷰를 삼갔다. 결국 베르트랑 들라노에의 주장에 대한 조사는 체계적으로 이루어지지는 않았다. 2006년 동계 올림픽을 유치했던 토리노도 이 논쟁에서 빠져나갈 수 없었다. 이번에는 스위스 국적의 IOC위원 마크 호들러(Marc Hodler)가 이 논쟁의 중심이 되었는데, 이 위원은 스위스 시온의 경쟁 도시였던", "BBC 다큐멘터리인 '파노라마'에서는 '매수된 올림픽'이란 주제로 2004년 8월에 방송을 내보내기도 했다. 이때 이 프로그램에서는 2012년 하계 올림픽의 개최지 선정과 관련된 뇌물에 대해서 조사했다. 이 다큐멘터리에서는 특정 후보 도시가 IOC 위원들에게 뇌물수수하는 것이 가능했다고 주장했으며, 특히 파리 시장이었던 베르트랑 들라노에(Bertrand Delanoë)는 영국의 총리인 토니 블레어와 런던올림픽유치위원회가 입후보 규정을 위반했다고 비난했다. 그는 당시 프랑스 대통령이었던 자크 시라크를 목격자로 내세웠지만 시라크 대통령은 이 분쟁에 휘말려드는 것을 주의했으며 인터뷰를 삼갔다. 결국 베르트랑 들라노에의 주장에 대한 조사는 체계적으로 이루어지지는 않았다. 2006년 동계 올림픽을 유치했던 토리노도 이 논쟁에서 빠져나갈 수 없었다. 이번에는 스위스 국적의 IOC위원 마크 호들러(Marc Hodler)가 이 논쟁의 중심이 되었는데, 이 위원은 스위스 시온의 경쟁 도시였던",

# HyDE (Hypothetical Document Embedding)
## 개념
- 질문(query)에 대한 가상의 답변 문서를 생성하고, 이 생성된 가상의 답변 문서를 임베딩하여 검색에 활용하는 기법이다.
- 일반적인 RAG에서 검색은 질문을 임베딩하여 문서 임베딩과 직접 비교한다. 
- 질문("파리는 어떤 도시인가?")과 답변 문서("파리는 프랑스의 수도이며...")는 표현 방식이 다르다는 문제가 있다. 
- 즉 의미적으로는 관련있지만 벡터 공간(임베딩 벡터간의 유사서)에서는 거리가 멀 수가 있다.
- 그래서 HyDE는 질문과 문서가 아니라 질문으로 가상의 답변 문서를 만들고 **가상의 답변문서와 저장된 문서들간의 유사도**를 비교한다.

## HyDE 프로세스

1. 가상 문서 생성
   -  LLM을 사용해 질문에 대한 가상의 답변 문서 생성
   -  이때 성능이 좋은 LLM을 사용하는 것이 좋다.
2. 임베딩 변환
   - `1`에서 생성한 가상 문서를 벡터로 임베딩
3. 유사도 검색
   - 가상 문서 임베딩으로 Vector Store에 저장된 문서들 중 유사한 문서를 검색
4. 답변 생성
   - 검색된 실제 문서를 바탕으로 최종 답변 생성

## 장단점
- **장점**
  - 질문-문서 간 의미적 차이를 해결 해서 정확한 문서 검색 가능
  - 질문 표현 방식에 덜 민감
- **단점**
  - 가상 문서의 품질에 따른 성능 편차가 발생한다.
    - 가상 문서 생성 시 환각(hallucination) 위험
  - 추가적인 LLM 호출로 인한 비용 증가한다.

#  MultiQueryRetriever

## 개념
- 하나의 사용자 질문으로 **여러 개의 다양한 질문을 생성하여 검색**을 수행하는 방법이다.
- 단일 질문의 한계를 극복하고 다각도에서 관련 정보 검색할 수있다.
  - 기본 RAG는 사용자의 질문의 질(quality)에 따라 검색 결과가 좌우된다.
  - 사용자가 한 질문에만 의존하는 것이 아니라 그 질문을 바탕으로 **다양한 의미의 질문들을 생성해서 단일 질문이 가지는 표현의 한계를 보완**한다.
    - 동일한 질문을 다른 각도에서 접근할 수있다.
    - 다양한 어휘와 표현으로 질문을 재구성한다.
- 예)
  - **원본 질문**: "딥러닝의 장점은 무엇인가?"
  - **생성된 질문들**:
    1. "딥러닝이 전통적인 머신러닝보다 나은 점은?"
    2. "딥러닝을 사용하면 얻을 수 있는 이익은?"
    3. "딥러닝의 주요 강점과 특징은?"
    4. "딥러닝 기술의 핵심 우위는?"
## 실행 프로세스

1. 질문 생성   
   - LLM을 사용해 원본 질문을 3-5개의 서로 다른 질문으로 변환
2. 병렬 검색
   - 생성된 각 질문으로 독립적으로 문서 검색 수행
3. 결과 통합
   - 여러 검색 결과를 하나로 병합
4. 중복 제거
   - 동일한 문서가 여러 번 검색된 경우 중복 제거
5. 최종 답변
   - 통합된 문서 세트를 바탕으로 답변 생성

## 장단점
- **장점**
    - 단일 질문으로 놓칠 수 있는 관련 문서 발견 수 있다.
    - 사용자 질문 표현 방식의 한계 극복
    - 더 포괄적이고 완전한 정보 검색 및 수집을 할 수있다.
- **단점**
    - 여러 번의 LLM 호출과 검색 수행이 실행 되므로 **계산비용, 토큰비용, 응답시간이 증가한다.**
    - 생성된 질문의 품질에 따른 성능 편차가 있을 수 있다.
    - 생성된 질문에 때라 원래 질문과 관련성 낮은 문서도 검색될 수 있어 최종 답변을 방해하는 노이즈가 증가할 수있다.

# MapReduce RAG 방식

## 개요

- RAG(Retrieval-Augmented Generation)에서 검색된 문서들 중 질문과 관련성이 높은 문서만을 선별하여 더 정확한 답변을 생성하는 방법이다.
- 검색된 문서들 중에서 질문 답변에 실제로 도움이 되는 문서만을 LLM을 통해 선별한 후 전달하는 방식이다.

## MapReduce 방식 프로세스
1. Map (문서 검색)
  - 벡터스토어에서 질문과 유사한 문서들을 의미적 유사도 검색으로 찾는다.
  - 이 단계에서는 단순 벡터 유사도만 고려하므로 질문과 직접적인 관련이 없는 문서도 포함될 수 있다.
2. Reduce (문서 선별 및 요약)
   - 검색된 각 문서가 질문 답변에 실제로 도움이 되는지 LLM에게 평가 요청한다.
   - 관련성이 높은 문서들만 선별하여 요약하거나 결합한다.
   - 필요시 여러 문서의 정보를 통합하여 더 응답에 적합한 컨텍스트를 생성한다.
3.  Generate (최종 답변 생성)
    - 질문과 선별된 컨텍스트를 함께 LLM에 전달하여 최종 답변을 생성한다.

## 장단점

- **장점**
  - **높은 정확도**: 질문과 직접 관련된 정보만 사용하여 더 정확한 답변을 생성한다.
  - **노이즈 제거**: 유사하지만 관련 없는 정보로 인한 혼동을 방지한다.
  - **컨텍스트 최적화**: 제한된 토큰 범위 내에서 가장 유용한 정보만 전달한다.
  - **확장성**: 많은 문서가 검색되어도 중요한 정보만 선별하여 처리할 수 있다.
- 단점
  - **추가 비용**: 문서 선별을 위한 LLM 호출로 인한 비용이 증가한다.
  - **처리 시간**: 문서 평가 단계가 추가되어 응답 속도가 저하된다.
  - **복잡성**: 구현과 관리가 더 복잡하다.
  - **의존성**: 문서 선별 성능이 LLM의 판단 능력에 크게 의존한다.